# Mortality EDA – Cluj-Napoca (UAT 54975)

**Climate-first approach**: load the processed climate data (with heatwave flags) and
the individual mortality records, aggregate deaths to daily counts, and overlay the
two to visualise the heatwave–death correlation.

Data period: determined by `ANALYSIS_PERIOD` constant (overlap of climate and mortality sources).

In [ ]:
%pip install -r ../requirements.txt

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname("__file__"), ".."))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from constants import (
    FIRST_PHASE_DIR, UAT_ID_CLUJ_NAPOCA,
    FIGURE_WIDTH, FIGURE_HEIGHT, FONT_SIZE, PLT_STYLE,
    MORTALITY_RISK_THRESHOLD, UTCI_HEAT_STRESS, TX_EXTREME,
    HUMIDITY_HUMID_HEAT, HUMIDITY_DRY_HEAT,
    TOP_N_DAYS, NUMERIC_COLS,
    SUMMER_MONTHS, AGE_BINS, AGE_LABELS, MORTALITY_COLS,
    ANALYSIS_PERIOD,
)

plt.style.use(PLT_STYLE)
plt.rcParams['figure.figsize'] = (FIGURE_WIDTH, FIGURE_HEIGHT)
plt.rcParams['font.size'] = FONT_SIZE
sns.set_palette('husl')

## 1  Load & prepare data

In [ ]:
climate = pd.read_csv(
    FIRST_PHASE_DIR / f'processed_UAT_{UAT_ID_CLUJ_NAPOCA}.csv',
    parse_dates=['date'],
)
for col in ['heat_stress_day', 'humid_heat', 'dry_heat']:
    climate[col] = climate[col].astype(str).str.strip().str.lower() == 'true'

print(f'Climate rows : {len(climate):,}  ({climate["date"].min().date()} - {climate["date"].max().date()})')
climate.head()

In [ ]:
mort_raw = pd.read_csv(
    FIRST_PHASE_DIR / f'mortality_UAT_{UAT_ID_CLUJ_NAPOCA}.csv',
    usecols=['DAT_DEC', 'DAT_NAS', 'SEX', 'CODB', 'source_year'],
    dtype={'SEX': 'Int64', 'CODB': str, 'source_year': 'Int64'},
)
mort_raw['DAT_DEC'] = pd.to_datetime(mort_raw['DAT_DEC'], errors='coerce')
mort_raw['DAT_NAS'] = pd.to_datetime(mort_raw['DAT_NAS'], errors='coerce')

n_before = len(mort_raw)
mort_raw = mort_raw.dropna(subset=['DAT_DEC'])
print(f'Mortality rows: {len(mort_raw):,}  (dropped {n_before - len(mort_raw)} with null DAT_DEC)')
print(f'Date range    : {mort_raw["DAT_DEC"].min().date()} - {mort_raw["DAT_DEC"].max().date()}')
mort_raw.head()

In [ ]:
# ── Aggregate to daily death counts ──────────────────────────────────────────
daily_deaths = (
    mort_raw
    .groupby(mort_raw['DAT_DEC'].dt.normalize())
    .size()
    .rename('death_count')
    .reset_index()
    .rename(columns={'DAT_DEC': 'date'})
)

# ── Join climate + daily deaths on date ──────────────────────────────────────
# Keep only the overlapping period (mortality years)
date_min = daily_deaths['date'].min()
date_max = daily_deaths['date'].max()
df = climate[(climate['date'] >= date_min) & (climate['date'] <= date_max)].copy()
df = df.merge(daily_deaths, on='date', how='left')
df['death_count'] = df['death_count'].fillna(0).astype(int)

print(f'Joined rows   : {len(df):,}  ({date_min.date()} - {date_max.date()})')
print(f'Total deaths  : {df["death_count"].sum():,}')
df.head()

## 2  Daily death-count time series

In [ ]:
fig, ax = plt.subplots(figsize=(FIGURE_WIDTH, FIGURE_HEIGHT))

ax.plot(df['date'], df['death_count'], linewidth=0.4, alpha=0.5, label='Daily')
ax.plot(df['date'], df['death_count'].rolling(7, center=True).mean(),
        linewidth=1.0, label='7-day avg')
ax.plot(df['date'], df['death_count'].rolling(30, center=True).mean(),
        linewidth=1.5, color='crimson', label='30-day avg')

ax.set_xlabel('Date')
ax.set_ylabel('Deaths per day')
ax.set_title(f'Daily death counts – Cluj-Napoca ({ANALYSIS_PERIOD})')
ax.legend()
plt.tight_layout()
plt.show()

## 3  Monthly seasonality

In [ ]:
monthly = df.copy()
monthly['month'] = monthly['date'].dt.month
monthly_totals = monthly.groupby('month')['death_count'].sum()

colours = ['#d62728' if m in SUMMER_MONTHS else '#1f77b4' for m in monthly_totals.index]

fig, ax = plt.subplots(figsize=(FIGURE_WIDTH, FIGURE_HEIGHT))
ax.bar(monthly_totals.index, monthly_totals.values, color=colours)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                     'Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_xlabel('Month')
ax.set_ylabel(f'Total deaths ({ANALYSIS_PERIOD})')
ax.set_title('Monthly seasonality – Cluj-Napoca (summer months highlighted in red)')
plt.tight_layout()
plt.show()

## 4  Deaths per year

In [ ]:
yearly = df.copy()
yearly['year'] = yearly['date'].dt.year
yearly_totals = yearly.groupby('year')['death_count'].sum()

fig, ax = plt.subplots(figsize=(FIGURE_WIDTH, FIGURE_HEIGHT))
ax.bar(yearly_totals.index, yearly_totals.values, color='steelblue')
ax.set_xlabel('Year')
ax.set_ylabel('Total deaths')
ax.set_title('Annual death totals – Cluj-Napoca')
for yr, val in yearly_totals.items():
    ax.text(yr, val + 20, str(val), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 5  Age-at-death distribution

In [ ]:
mort_age = mort_raw.dropna(subset=['DAT_NAS']).copy()
mort_age['age'] = (mort_age['DAT_DEC'] - mort_age['DAT_NAS']).dt.days / 365.25
mort_age['age_group'] = pd.cut(mort_age['age'], bins=AGE_BINS, labels=AGE_LABELS, right=False)

# Map SEX codes to labels
sex_map = {1: 'Male', 2: 'Female'}
mort_age['sex_label'] = mort_age['SEX'].map(sex_map).fillna('Unknown')

fig, axes = plt.subplots(1, 2, figsize=(FIGURE_WIDTH, FIGURE_HEIGHT))

# Histogram + KDE
for sex_label, colour in [('Male', '#1f77b4'), ('Female', '#d62728')]:
    subset = mort_age[mort_age['sex_label'] == sex_label]['age']
    axes[0].hist(subset, bins=60, alpha=0.5, density=True, label=sex_label, color=colour)
    subset.plot.kde(ax=axes[0], color=colour, linewidth=1.5)
axes[0].set_xlabel('Age at death')
axes[0].set_ylabel('Density')
axes[0].set_title('Age-at-death distribution by sex')
axes[0].legend()
axes[0].set_xlim(0, 110)

# Age-group bar chart
age_counts = mort_age.groupby(['age_group', 'sex_label']).size().unstack(fill_value=0)
age_counts.plot.bar(ax=axes[1], color=['#d62728', '#1f77b4'])
axes[1].set_xlabel('Age group')
axes[1].set_ylabel('Count')
axes[1].set_title('Deaths by age group & sex')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f'Mean age at death: {mort_age["age"].mean():.1f}  |  Median: {mort_age["age"].median():.1f}')

## 6  Sex breakdown

In [ ]:
sex_counts = mort_raw['SEX'].map({1: 'Male', 2: 'Female'}).value_counts()

fig, ax = plt.subplots(figsize=(8, FIGURE_HEIGHT))
sex_counts.plot.bar(ax=ax, color=['#1f77b4', '#d62728'])
ax.set_xlabel('Sex')
ax.set_ylabel('Total deaths')
ax.set_title(f'Deaths by sex – Cluj-Napoca ({ANALYSIS_PERIOD})')
for i, (label, val) in enumerate(sex_counts.items()):
    ax.text(i, val + 50, f'{val:,}', ha='center', fontsize=11)
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## 7  Heatwave overlay (key plot)

Dual-axis time series: daily death count (left axis) vs. UTCI_MX (right axis),
with shaded bands for `heat_stress_day == True` periods.

In [ ]:
def plot_heatwave_overlay(data, year=None, title_suffix=''):
    """Dual-axis overlay of deaths and UTCI_MX with heat-stress shading."""
    if year is not None:
        mask = (data['date'].dt.year == year) & (data['date'].dt.month.isin(SUMMER_MONTHS))
        sub = data[mask].copy()
        period = f'Summer {year}'
    else:
        sub = data[data['date'].dt.month.isin(SUMMER_MONTHS)].copy()
        period = 'All summers (Jun–Aug)'

    fig, ax1 = plt.subplots(figsize=(FIGURE_WIDTH, FIGURE_HEIGHT + 1))

    # Shade heat-stress days
    heat_days = sub[sub['heat_stress_day']]
    for _, row in heat_days.iterrows():
        ax1.axvspan(row['date'] - pd.Timedelta(hours=12),
                    row['date'] + pd.Timedelta(hours=12),
                    alpha=0.15, color='red', zorder=0)

    # Left axis — death count
    colour_deaths = '#1f77b4'
    ax1.bar(sub['date'], sub['death_count'], width=1, alpha=0.7,
            color=colour_deaths, label='Deaths / day')
    ax1.set_ylabel('Deaths per day', color=colour_deaths)
    ax1.tick_params(axis='y', labelcolor=colour_deaths)

    # Right axis — UTCI_MX
    ax2 = ax1.twinx()
    colour_utci = '#d62728'
    ax2.plot(sub['date'], sub['UTCI_MX'], color=colour_utci,
             linewidth=1.2, alpha=0.9, label='UTCI_MX')
    ax2.axhline(UTCI_HEAT_STRESS, color='orange', linestyle='--',
                linewidth=0.8, label=f'Heat-stress threshold ({UTCI_HEAT_STRESS}°C)')
    ax2.set_ylabel('UTCI_MX (°C)', color=colour_utci)
    ax2.tick_params(axis='y', labelcolor=colour_utci)

    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
    ax1.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
    fig.autofmt_xdate(rotation=45)

    # Combined legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

    ax1.set_title(f'Heatwave overlay – Cluj-Napoca – {period}{title_suffix}')
    plt.tight_layout()
    plt.show()

    return sub

In [ ]:
# Plot each summer individually for readability
for yr in sorted(df['date'].dt.year.unique()):
    plot_heatwave_overlay(df, year=yr)

## 8  Heatwave vs. non-heatwave box-plot

In [ ]:
# Build a categorical column for the box-plot
conditions = []
for _, row in df.iterrows():
    if row['humid_heat']:
        conditions.append('Humid heat')
    elif row['dry_heat']:
        conditions.append('Dry heat')
    elif row['heat_stress_day']:
        conditions.append('Heat stress (other)')
    else:
        conditions.append('Normal')
df['heat_category'] = pd.Categorical(
    conditions,
    categories=['Normal', 'Heat stress (other)', 'Dry heat', 'Humid heat'],
    ordered=True,
)

df['heat_label'] = df['heat_stress_day'].map({True: 'Heat stress', False: 'Normal'})

fig, axes = plt.subplots(1, 2, figsize=(FIGURE_WIDTH, FIGURE_HEIGHT))

# (a) Heat-stress vs normal
sns.boxplot(data=df, x='heat_label', y='death_count', hue='heat_label',
            ax=axes[0], palette={'Normal': '#1f77b4', 'Heat stress': '#d62728'},
            order=['Normal', 'Heat stress'], legend=False)
axes[0].set_xlabel('')
axes[0].set_ylabel('Deaths per day')
axes[0].set_title('Heat-stress day vs. normal')

# (b) Four-way breakdown
sns.boxplot(data=df, x='heat_category', y='death_count', hue='heat_category',
            ax=axes[1], palette=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'],
            legend=False)
axes[1].set_xlabel('')
axes[1].set_ylabel('Deaths per day')
axes[1].set_title('Deaths by heat condition')
axes[1].tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.show()

# Summary stats
print(df.groupby('heat_stress_day')['death_count'].describe().round(2))
print()
print(df.groupby('heat_category')['death_count'].describe().round(2))

## 9  Summer excess mortality

In [ ]:
df['year'] = df['date'].dt.year
df['is_summer'] = df['date'].dt.month.isin(SUMMER_MONTHS)

summer_stats = (
    df.groupby(['year', 'is_summer'])['death_count']
    .mean()
    .unstack(fill_value=0)
    .rename(columns={False: 'rest_of_year', True: 'summer'})
)
summer_stats['excess_ratio'] = summer_stats['summer'] / summer_stats['rest_of_year']
summer_stats['excess_diff']  = summer_stats['summer'] - summer_stats['rest_of_year']

fig, axes = plt.subplots(1, 2, figsize=(FIGURE_WIDTH, FIGURE_HEIGHT))

# (a) Mean daily deaths: summer vs rest
summer_stats[['rest_of_year', 'summer']].plot.bar(ax=axes[0],
    color=['#1f77b4', '#d62728'])
axes[0].set_ylabel('Mean daily deaths')
axes[0].set_title('Mean daily deaths: summer vs rest-of-year')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(['Rest of year', 'Summer (Jun–Aug)'])

# (b) Excess ratio
axes[1].bar(summer_stats.index, summer_stats['excess_ratio'],
            color=['#d62728' if r > 1 else '#1f77b4' for r in summer_stats['excess_ratio']])
axes[1].axhline(1.0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_ylabel('Summer / rest-of-year ratio')
axes[1].set_title('Summer excess mortality ratio')
axes[1].set_xlabel('Year')
for yr, val in summer_stats['excess_ratio'].items():
    axes[1].text(yr, val + 0.005, f'{val:.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print(summer_stats.round(3))

## 10  Top-N deadliest days

In [ ]:
top_cols = ['date', 'death_count', 'TX', 'UTCI_MX', 'RH_AVG',
            'heat_stress_day', 'humid_heat', 'dry_heat']
top_days = df.nlargest(TOP_N_DAYS, 'death_count')[top_cols].reset_index(drop=True)
top_days['date'] = top_days['date'].dt.strftime('%Y-%m-%d')

print(f'Top {TOP_N_DAYS} deadliest days in Cluj-Napoca ({ANALYSIS_PERIOD}):\n')
top_days

---

### Summary

The plots above apply the **Climate-first** approach: heatwave windows (identified
via `heat_stress_day`, `humid_heat`, `dry_heat` flags from the processed climate
data) are overlaid on daily death counts to visually and statistically assess
whether mortality spikes align with extreme heat periods in Cluj-Napoca.

**Next steps:**
- Formal statistical testing (e.g. Welch's t-test, Mann–Whitney U) on heatwave vs. baseline deaths.
- Lag analysis (deaths may peak 1–3 days after the heat peak).
- Cause-of-death filtering (cardiovascular/respiratory via `CODB` ICD codes).